## EEG BrainLat — Download do Dataset
### 1_Download_Datatset

---

**Objetivo:** baixar o BrainLat via Synapse e validar a integridade básica dos arquivos.

---

#### Estrutura do notebook

| Fase | Descrição |
|------|-----------|
| **0 — Configuração** | Imports globais e definições de caminhos |
| **1 — Download** | Sinapse: AD e HC |
| **1.1 — Verificação** | Checagem de tamanho e arquivos baixados |

---


---
### Phase 0 — Global Configuration

All imports and constants are centralised here.
**Execute this phase before any other.**


In [ ]:
# 0.1 -- Global Imports
import os
import re
import warnings
import traceback
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mne
from mne.time_frequency import psd_array_welch

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.inspection import PartialDependenceDisplay
from xgboost import XGBClassifier
from scipy.stats import mannwhitneyu, spearmanr

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed. Run: pip install shap")

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

SEED = 42
np.random.seed(SEED)
print("Imports loaded successfully.")


In [ ]:
# 0.2 -- Path Configuration and Constants
#
# Adapt the paths below to your environment.
# Only modify this cell; all subsequent cells use these variables.

ROOT_DIR      = "./Dataset_EEG_Alzheimer"
DIR_AD        = os.path.join(ROOT_DIR, "dataset_eeg_alzheimer")
DIR_HC        = os.path.join(ROOT_DIR, "dataset_eeg_hc")

# Synapse folder IDs (do not modify)
SYNAPSE_ID_AD = "syn53222482"
SYNAPSE_ID_HC = "syn53222486"

# Output files
OUT_CSV_FULL   = "eeg_features_brainlat_FULL.csv"
OUT_CSV_FAILED = "eeg_features_brainlat_failures.csv"

# Frequency bands (Hz)
BANDS = {
    "Delta": (0.5,  4.0),
    "Theta": (4.0,  8.0),
    "Alpha": (8.0, 13.0),
    "Beta" : (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}
FMIN, FMAX = 0.5, 45.0

# Feature set used in the ML model
# Rel_Delta_mean is excluded: near-zero variance in this dataset (dead feature)
FEATURE_COLS = [
    "Rel_Theta_mean",
    "Rel_Alpha_mean",
    "Rel_Beta_mean",
    "Rel_Gamma_mean",
    "Razao_Theta_Alpha",
    "Razao_Theta_Beta",
    "Spectral_Entropy",
]

print("Configuration loaded.")
print(f"  AD directory : {DIR_AD}")
print(f"  HC directory : {DIR_HC}")
print(f"  Features     : {FEATURE_COLS}")


---
### Phase 1 -- Data Download (BrainLat / Synapse)

The BrainLat dataset is hosted on [Synapse](https://www.synapse.org/).
To download it:

1. Create a free account at [synapse.org](https://www.synapse.org/).
2. Accept the dataset terms of use.
3. Generate a **Personal Access Token** under: `Profile > Personal Access Token > Create`.
4. Paste the token into `SYNAPSE_TOKEN` below.

> **Security:** Never commit your token to a public repository.
> Add your token file to `.gitignore`.


In [ ]:
# 1.1 -- Download AD group
import synapseclient
import synapseutils

# Replace with your Synapse Personal Access Token
SYNAPSE_TOKEN = "YOUR_SYNAPSE_TOKEN_HERE"

syn = synapseclient.Synapse()
syn.login(authToken=SYNAPSE_TOKEN)

os.makedirs(DIR_AD, exist_ok=True)
print(f"Downloading AD group (ID: {SYNAPSE_ID_AD})...")
synapseutils.syncFromSynapse(syn, SYNAPSE_ID_AD, path=DIR_AD)
print("AD download complete.")


In [ ]:
# 1.2 -- Download HC group
os.makedirs(DIR_HC, exist_ok=True)
print(f"Downloading HC group (ID: {SYNAPSE_ID_HC})...")
synapseutils.syncFromSynapse(syn, SYNAPSE_ID_HC, path=DIR_HC)
print("HC download complete.")


In [ ]:
# 1.3 -- Dataset Size Verification
# Handles both .set-only (AD) and .set+.fdt (HC) formats.

print("=" * 55)
print("  EEG DATA SIZE VERIFICATION")
print("=" * 55)

size_counts = {"AD": [], "HC": []}

for group, folder in [("AD", DIR_AD), ("HC", DIR_HC)]:
    for root, _, files in os.walk(folder):
        for fname in files:
            if fname.endswith(".set"):
                path_set = os.path.join(root, fname)
                size_bytes = os.path.getsize(path_set)
                path_fdt = path_set.replace(".set", ".fdt")
                if os.path.exists(path_fdt):
                    size_bytes += os.path.getsize(path_fdt)
                size_counts[group].append((fname, size_bytes / (1024 ** 2)))

for group, items in size_counts.items():
    print(f"\n{group}: {len(items)} complete recordings found")
    for name, mb in items[:5]:
        print(f"  {name}  ({mb:.1f} MB)")
    if len(items) > 5:
        print(f"  ... and {len(items) - 5} more file(s).")

all_ad = [mb for _, mb in size_counts["AD"]]
all_hc = [mb for _, mb in size_counts["HC"]]
all_sizes = all_ad + all_hc

if all_sizes:
    fig, ax = plt.subplots(figsize=(9, 4))
    bins = np.linspace(min(all_sizes), max(all_sizes), 15)
    if all_ad:
        ax.hist(all_ad, bins=bins, alpha=0.7, color="#d62728", label="AD")
    if all_hc:
        ax.hist(all_hc, bins=bins, alpha=0.7, color="#2ca02c", label="HC")
    ax.set_xlabel("Total on-disk size per subject (MB)")
    ax.set_ylabel("Number of subjects")
    ax.set_title("EEG Data Size Distribution by Group")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No data found. Verify DIR_AD and DIR_HC.")
